# 07 — Compressor Development Notebook

**Purpose:** Walk a developer through building a new compressor from scratch, running the full test suite, and manually inspecting compression quality on realistic planning contexts.

---

## How to use this notebook

| Goal | What to do |
|---|---|
| Build a non-trainable compressor | Work through Sections 2–3, run tests in Section 5 |
| Build a trainable compressor (PPO-compatible) | Complete Sections 2–4, run tests in Section 5 |
| Inspect compression quality | Jump to Section 6 and load any registered compressor |
| Run full test suite only | Run Sections 1 + 5 |

**Do not skip Section 1** — it wires the Python path and loads fixtures used by later cells.

---

## Architecture cheat-sheet

```
CompressorBase (ABC)
│   compress(trajectory, previous_state=None) → CompressedState  [abstract]
│   get_log_probs(traj_text, comp_text) → Tensor                 [raises by default]
│   get_trainable_parameters() → list                            [returns []]
│   is_trainable() → bool                                        [len(params) > 0]
│   get_metadata() → dict                                        [base impl provided]
│
└── TrainableCompressorBase (ABC)
        get_log_probs(...)          [abstract — must return grad-connected Tensor]
        get_trainable_parameters()  [abstract — return list[nn.Parameter]]
        save_checkpoint(path)       [abstract]
        load_checkpoint(path)       [abstract — raise CompressorCheckpointError on fail]
        apply_lora(lora_config)     [concrete — delegates to lora_utils]
        freeze_base_layers(freeze)  [concrete]
```

**CompressedState** — the output contract. Six required sections:

| Section | Type | Role |
|---|---|---|
| `hard_constraint_ledger` | `HardConstraintLedger` | Which hard constraints are satisfied/violated/unknown |
| `soft_constraints_summary` | `str` | Compressed description of soft preferences |
| `decisions_made` | `list[str]` | Irreversible committed actions (bookings) |
| `open_questions` | `list[str]` | Planning gaps still to resolve |
| `key_discoveries` | `list[str]` | Prices, options, facts gathered so far |
| `current_itinerary_sketch` | `str` | Day-by-day partial plan |

The `CompressedStateTemplate` renders these sections into the `## SECTION_NAME ##` text format
that gets injected into the agent's LLM context window.

**Key invariant:** `get_log_probs()` must return a 1-D tensor that is connected to the
autograd graph, so that `(-log_probs.mean()).backward()` populates all trainable parameter
gradients. Violating this breaks PPO training silently.


## Section 1 — Setup

In [1]:
# ── Standard library + path wiring ─────────────────────────────────────────────
import sys
import json
import uuid
import math
import tempfile
from pathlib import Path

# Ensure the src/ directory is importable regardless of how the notebook was launched.
# This one-liner is safer than manipulating PYTHONPATH in the shell.
REPO_ROOT = Path("..").resolve()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import torch
import torch.nn as nn

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"Repo root       : {REPO_ROOT}")

PyTorch version : 2.11.0
CUDA available  : False
Repo root       : /Users/kyledennis/Documents/COLUMBIA SPRING 2026/Generative AI /Project/optimized-llm-planning-memory


In [2]:
# ── Project imports ─────────────────────────────────────────────────────────────
from optimized_llm_planning_memory.compressor.base import CompressorBase
from optimized_llm_planning_memory.compressor.trainable_base import TrainableCompressorBase
from optimized_llm_planning_memory.compressor.template import CompressedStateTemplate
from optimized_llm_planning_memory.compressor.identity_compressor import IdentityCompressor
from optimized_llm_planning_memory.compressor.dummy_compressor import DummyCompressor
from optimized_llm_planning_memory.core.models import (
    CompressedState,
    HardConstraintLedger,
    ReActStep,
    ToolCall,
    ToolResult,
    TrajectoryModel,
)
from optimized_llm_planning_memory.core.exceptions import (
    CompressorCheckpointError,
    LogProbsNotSupportedError,
)

# ── Template singleton — reuse across the notebook ─────────────────────────────
TEMPLATE = CompressedStateTemplate()

# ── Dev context directory ───────────────────────────────────────────────────────
CONTEXTS_DIR = REPO_ROOT / "data" / "compressor_dev" / "contexts"
print(f"\nContext files available:")
for f in sorted(CONTEXTS_DIR.glob("*.json")):
    print(f"  {f.name}")


Context files available:
  01_early_episode.json
  02_mid_episode.json
  03_late_episode.json
  04_stress_context.json


In [3]:
# ── Utility: load a dev context JSON → TrajectoryModel ─────────────────────────
def load_dev_context(filename: str) -> tuple[TrajectoryModel, dict]:
    """
    Load one of the pre-made compressor dev contexts.

    Returns
    -------
    (trajectory_model, metadata)  where metadata contains description,
    request_summary, and compression_challenge from the JSON.
    """
    path = CONTEXTS_DIR / filename
    raw = json.loads(path.read_text(encoding="utf-8"))

    # Build TrajectoryModel from raw steps
    steps = []
    for s in raw["trajectory"]["steps"]:
        action_raw = s["action"]
        obs_raw = s["observation"]
        step = ReActStep(
            step_index=s["step_index"],
            thought=s["thought"],
            action=ToolCall(
                tool_name=action_raw["tool_name"],
                arguments=action_raw["arguments"],
                call_id=action_raw.get("call_id", ""),
                raw_text=json.dumps(action_raw),
            ),
            observation=ToolResult(
                tool_name=action_raw["tool_name"],
                success=obs_raw["success"],
                result=obs_raw.get("result"),
                error_message=obs_raw.get("error_message"),
                latency_ms=obs_raw.get("latency_ms", 0.0),
            ),
            # Dev context fixtures use a simplified dict format for itinerary_snapshot
            # that does not match the full Itinerary Pydantic model, so we pass None.
            # Booking info is still accessible via step observations.
            itinerary_snapshot=None,
            timestamp=None,
        )
        steps.append(step)

    traj = TrajectoryModel(
        trajectory_id=raw["trajectory"]["trajectory_id"],
        request_id=raw["trajectory"]["request_id"],
        steps=tuple(steps),
        total_steps=raw["trajectory"]["total_steps"],
    )
    metadata = {
        "description": raw["description"],
        "request_summary": raw["request_summary"],
        "compression_challenge": raw["compression_challenge"],
    }
    return traj, metadata


# ── Utility: pretty-print a CompressedState ────────────────────────────────────
def show_compression_result(state: CompressedState, compressor_name: str = "") -> None:
    label = f"[{compressor_name}]" if compressor_name else ""
    print(f"\n{'='*70}")
    print(f" COMPRESSED STATE {label}")
    print(f"  method     : {state.compression_method}")
    print(f"  step_index : {state.step_index}")
    print(f"  token_count: {state.token_count}")
    print(f"{'='*70}")
    rendered = TEMPLATE.render(state)
    print(rendered)
    print(f"{'='*70}")
    print(f"Total characters: {len(rendered)}")


print("Utilities loaded.")

Utilities loaded.


## Section 2 — Build a minimal non-trainable compressor

Start here to add a new rule-based or prompt-based compressor that does **not** need to
participate in the PPO training loop.

**Steps:**
1. Subclass `CompressorBase` (not `TrainableCompressorBase`).
2. Implement `compress()` — return a fully-populated `CompressedState`.
3. Optionally override `get_metadata()` to add type-specific fields.
4. Test the forward pass in the cell below the class definition.

The example below builds `KeywordSummaryCompressor`: a simple rule-based compressor
that extracts keyword summaries from each step's thought and observation.
It is intentionally naive — the point is to demonstrate the contract, not to win benchmarks.

In [4]:
import re
from datetime import datetime, timezone


class KeywordSummaryCompressor(CompressorBase):
    """
    Rule-based compressor that summarises each step by extracting:
    - Tool name and success/failure from observations
    - Key numbers (prices, counts) from result text
    - Booking IDs from successful book_* calls

    Not trainable — subclasses CompressorBase directly.
    """

    # Pattern to find currency amounts and round numbers in JSON-like text
    _PRICE_RE = re.compile(r"\$?([\d,]+(?:\.\d{1,2})?)")
    _BOOKING_RE = re.compile(r'"booking_id"\s*:\s*"([^"]+)"')

    def compress(
        self,
        trajectory: TrajectoryModel,
        previous_state: CompressedState | None = None,
    ) -> CompressedState:
        decisions: list[str] = []
        discoveries: list[str] = []
        open_qs: list[str] = []

        for step in trajectory.steps:
            obs = step.observation
            tool = step.action.tool_name if step.action else "unknown"
            result_text = json.dumps(obs.result) if obs and obs.result else ""

            if obs and not obs.success:
                # Failed calls are open questions, not decisions
                open_qs.append(
                    f"step {step.step_index}: {tool} FAILED — {obs.error_message or 'unknown error'}"
                )
                continue

            if tool.startswith("book_") or tool.startswith("add_"):
                # Extract booking ID if present
                bkg_match = self._BOOKING_RE.search(result_text)
                bkg_id = bkg_match.group(1) if bkg_match else "?"
                decisions.append(f"step {step.step_index}: {tool} → booking_id={bkg_id}")
            elif tool.startswith("search_") or tool == "get_available_routes":
                # Extract key numbers for discoveries
                prices = self._PRICE_RE.findall(result_text)[:3]  # at most 3 prices
                price_hint = ", ".join(f"${p}" for p in prices) if prices else "(no prices)"
                discoveries.append(f"step {step.step_index}: {tool} found options [{price_hint}]")

        # Build a minimal itinerary sketch from the most recent snapshot
        sketch = "No bookings yet."
        for step in reversed(trajectory.steps):
            if step.itinerary_snapshot:
                snap = step.itinerary_snapshot
                cost = snap.get("total_committed_cost", 0)
                remaining = snap.get("remaining_budget", "?")
                bookings = snap.get("bookings", [])
                lines = [f"Committed: ${cost:.2f}  |  Remaining: ${remaining}"]
                for b in bookings:
                    lines.append(f"  - [{b['type']}] {b['name']} (${b['cost']:.2f})")
                sketch = "\n".join(lines)
                break

        # Build an empty constraint ledger (rule-based: we don't parse constraints)
        ledger = HardConstraintLedger(
            constraints=[],
            satisfied_ids=[],
            violated_ids=[],
            unknown_ids=[],
        )

        full_text = f"{' | '.join(decisions + discoveries + open_qs)}\n{sketch}"

        return CompressedState(
            state_id=str(uuid.uuid4()),
            trajectory_id=trajectory.trajectory_id,
            step_index=trajectory.total_steps,
            hard_constraint_ledger=ledger,
            soft_constraints_summary="Keyword extraction: no soft-constraint parsing.",
            decisions_made=decisions if decisions else ["No confirmed bookings yet."],
            open_questions=open_qs if open_qs else ["None identified."],
            key_discoveries=discoveries if discoveries else ["No search results yet."],
            current_itinerary_sketch=sketch,
            compression_method="keyword_summary",
            token_count=len(full_text.split()),
            created_at=datetime.now(tz=timezone.utc).isoformat(),
        )

    def get_metadata(self) -> dict:
        return {
            "type": "keyword_summary",
            "param_count": 0,
            "trainable": False,
        }


# ── Smoke test: verify the contract ────────────────────────────────────────────
kw_compressor = KeywordSummaryCompressor()

assert isinstance(kw_compressor, CompressorBase)
assert not kw_compressor.is_trainable()
assert kw_compressor.get_trainable_parameters() == []
print("KeywordSummaryCompressor instantiated and contract checks passed.")
print(f"Metadata: {kw_compressor.get_metadata()}")

KeywordSummaryCompressor instantiated and contract checks passed.
Metadata: {'type': 'keyword_summary', 'param_count': 0, 'trainable': False}


In [5]:
# ── Quick forward-pass sanity check ───────────────────────────────────────────
# Load the mid-episode context (6 steps, one failed call) and compress it.
traj_mid, meta_mid = load_dev_context("02_mid_episode.json")

print(f"Context: {meta_mid['description']}")
print(f"Challenge: {meta_mid['compression_challenge']}")

state = kw_compressor.compress(traj_mid)

# Validate that the template can render this state without raising
TEMPLATE.validate(state)

# Show the compressed output
show_compression_result(state, "KeywordSummaryCompressor")

Context: Mid-episode: 6 steps. Hotel booked. One failed restaurant search (wrong argument), corrected search succeeds. Agent has open work (attractions, events) and a partial itinerary snapshot.
Challenge: Context is growing. A good compressor must distinguish committed decisions (hotel booked at $356 total) from exploratory results (restaurants listed, not reserved), record the corrected failed call without cluttering the ledger, and flag the remaining open questions (attractions, events not yet searched).

 COMPRESSED STATE [KeywordSummaryCompressor]
  method     : keyword_summary
  step_index : 6
  token_count: 65
## HARD_CONSTRAINT_LEDGER ##
Constraints: 0 total
Satisfied: []
Violated: []
Unknown: []

## SOFT_CONSTRAINTS_SUMMARY ##
Keyword extraction: no soft-constraint parsing.

## DECISIONS_MADE ##
- step 2: book_hotel → booking_id=bkg-hotel-0042

## OPEN_QUESTIONS ##
- step 3: search_restaurants FAILED — Invalid cuisine value 'mediteranean'. Accepted values: ['mediterranean', 'l

### Developer checklist for non-trainable compressors

Before committing a new non-trainable compressor:

- [ ] `compress()` never raises on zero-step trajectories
- [ ] `compress()` produces a `CompressedState` that passes `TEMPLATE.validate()`
- [ ] `get_metadata()` returns a dict with at least `type`, `param_count`, `trainable` keys
- [ ] `get_metadata()` is JSON-serialisable (no tensors or non-primitive types)
- [ ] The compression output shrinks with context length — inspect with the stress context (04)

## Section 3 — Upgrade to a trainable compressor

To hook your compressor into the PPO training loop, subclass `TrainableCompressorBase`
instead. This requires implementing four additional abstract methods:

| Method | Requirement |
|---|---|
| `get_log_probs(traj_text, comp_text)` | 1-D tensor, connected to autograd, values ≤ 0 |
| `get_trainable_parameters()` | `list[nn.Parameter]` — fed to the optimizer |
| `save_checkpoint(path)` | Write weights to disk; must be loadable |
| `load_checkpoint(path)` | Raise `CompressorCheckpointError` on failure |

The cell below upgrades `KeywordSummaryCompressor` with a tiny learned embedding.
The embedding doesn't change the rule-based `compress()` logic — it only provides a
differentiable path so PPO can update *something*. This is the same pattern used by
`IdentityCompressor` (_dummy_param).

**For real use:** replace the rule-based `compress()` with an actual neural forward pass
(encoder-decoder, attention, etc.) and derive `get_log_probs()` from that model's output.

In [ ]:
class TrainableKeywordSummaryCompressor(TrainableCompressorBase):
    """
    Trainable variant of KeywordSummaryCompressor.

    Architecture
    ------------
    - A 16-dim learned embedding over ASCII characters (vocab size = 128).
    - A single linear projection from 16-dim to 1-dim (scalar log-prob offset).

    compress() is identical to KeywordSummaryCompressor — still rule-based.
    get_log_probs() runs the compressed text through the embedding + projection
    to produce per-token log-probability offsets.

    This is a minimal scaffold, not a production model. Swap the embedding +
    projection for a real seq2seq model when you're ready.
    """

    VOCAB = 128  # ASCII printable range
    EMBED_DIM = 16

    def __init__(self, device: str = "cpu") -> None:
        self._device = torch.device(device)

        # Character-level embedding: maps ASCII code → EMBED_DIM vector
        # nn.Embedding is nn.Module so its parameters are tracked by autograd.
        self._embedding = nn.Embedding(self.VOCAB, self.EMBED_DIM).to(self._device)

        # Linear projection from embed_dim → scalar log-prob offset per token
        self._proj = nn.Linear(self.EMBED_DIM, 1, bias=False).to(self._device)

        # Delegate rule-based compress() to parent class (composition, not inheritance)
        self._rule_compressor = KeywordSummaryCompressor()

    # ── CompressorBase ─────────────────────────────────────────────────────────

    def compress(
        self,
        trajectory: TrajectoryModel,
        previous_state: CompressedState | None = None,
    ) -> CompressedState:
        # Delegate to the rule-based compressor; no neural inference at compress time.
        state = self._rule_compressor.compress(trajectory, previous_state)
        # Override compression_method to reflect this class
        return state.model_copy(update={"compression_method": "trainable_keyword_summary"})

    # ── TrainableCompressorBase ────────────────────────────────────────────────

    def get_log_probs(self, trajectory_text: str, compressed_text: str) -> torch.Tensor:
        """
        Per-token log-prob via character embedding + linear projection.

        Each whitespace-split token is encoded as the mean of its character
        embeddings, then projected to a scalar. Softplus + log gives a valid
        (negative) log-probability offset.

        Shape: (num_tokens,)  — 1-D, connected to autograd.
        """
        tokens = compressed_text.split() or [compressed_text or "<empty>"]
        log_probs = []
        for token in tokens:
            # Encode each character as an ASCII id, clamp to vocab range
            char_ids = torch.tensor(
                [min(ord(c), self.VOCAB - 1) for c in token],
                dtype=torch.long,
                device=self._device,
            )
            # Mean character embedding for the token
            emb = self._embedding(char_ids).mean(dim=0)  # (EMBED_DIM,)
            # Project to scalar; use -softplus to keep the value ≤ 0
            lp = -torch.nn.functional.softplus(self._proj(emb)).squeeze()
            log_probs.append(lp)

        # Stack into a 1-D tensor — the autograd graph flows back to _embedding
        # and _proj through each token's lp.
        return torch.stack(log_probs)  # shape: (num_tokens,)

    def get_trainable_parameters(self) -> list[nn.Parameter]:
        # Return all parameters from both modules so the optimizer can update them.
        return list(self._embedding.parameters()) + list(self._proj.parameters())

    def save_checkpoint(self, path: str) -> None:
        save_dir = Path(path)
        save_dir.mkdir(parents=True, exist_ok=True)
        torch.save(
            {
                "embedding": self._embedding.state_dict(),
                "proj": self._proj.state_dict(),
            },
            save_dir / "kw_compressor.pt",
        )

    def load_checkpoint(self, path: str) -> None:
        p = Path(path)
        if p.is_dir():
            p = p / "kw_compressor.pt"
        if not p.exists():
            raise CompressorCheckpointError(f"Checkpoint not found at {p}")
        try:
            ckpt = torch.load(p, map_location=self._device)
            self._embedding.load_state_dict(ckpt["embedding"])
            self._proj.load_state_dict(ckpt["proj"])
        except Exception as exc:
            raise CompressorCheckpointError(f"Failed to load checkpoint: {exc}") from exc

    def get_metadata(self) -> dict:
        n_params = sum(p.numel() for p in self.get_trainable_parameters())
        return {
            "type": "trainable_keyword_summary",
            "param_count": n_params,
            "trainable": True,
            "embed_dim": self.EMBED_DIM,
            "device": str(self._device),
        }


print("TrainableKeywordSummaryCompressor defined.")

## Section 4 — Verify the trainable contract

These checks replicate what `tests/test_compressor/test_comprehensive.py` runs automatically.
Run them here interactively to confirm your implementation is correct before running pytest.

In [ ]:
# ── Instantiation and metadata ─────────────────────────────────────────────────
tkw = TrainableKeywordSummaryCompressor(device="cpu")

assert isinstance(tkw, CompressorBase)
assert isinstance(tkw, TrainableCompressorBase)
assert tkw.is_trainable()
assert len(tkw.get_trainable_parameters()) > 0
meta = tkw.get_metadata()
assert all(k in meta for k in ("type", "param_count", "trainable"))
print(f"Metadata: {meta}")

In [ ]:
# ── Forward pass ───────────────────────────────────────────────────────────────
traj_early, _ = load_dev_context("01_early_episode.json")
state = tkw.compress(traj_early)

assert isinstance(state, CompressedState)
TEMPLATE.validate(state)  # must not raise
print(f"compress() OK — method={state.compression_method}, tokens={state.token_count}")

In [ ]:
# ── Gradient flow ──────────────────────────────────────────────────────────────
# This is THE most important check for PPO compatibility.
# The loss must flow gradients back to ALL trainable parameters.

traj_text = traj_early.to_text()
comp_text = TEMPLATE.render(state)

log_probs = tkw.get_log_probs(traj_text, comp_text)

# 1. Shape: 1-D tensor
assert log_probs.dim() == 1, f"Expected 1-D tensor, got shape {log_probs.shape}"

# 2. Values: all ≤ 0 (log-probabilities)
assert (log_probs <= 0.0 + 1e-6).all(), "Some log-probs are positive — check softplus sign"

# 3. Gradient flow: backward() must populate all parameter grads
loss = -log_probs.mean()  # policy gradient loss (maximise log_prob = minimise negative)
loss.backward()

params = tkw.get_trainable_parameters()
no_grad = [i for i, p in enumerate(params) if p.grad is None]
if no_grad:
    print(f"WARNING: parameters at indices {no_grad} have no gradient!")
else:
    print(f"All {len(params)} trainable parameters received gradients. ✓")
    grad_norms = [p.grad.norm().item() for p in params]
    print(f"Gradient norms: {[f'{n:.4f}' for n in grad_norms]}")

In [ ]:
# ── Optimizer step ─────────────────────────────────────────────────────────────
# Verify that a single SGD step changes the parameter values.
import torch.optim as optim

tkw2 = TrainableKeywordSummaryCompressor(device="cpu")
opt = optim.SGD(tkw2.get_trainable_parameters(), lr=0.1)

before = [p.data.clone() for p in tkw2.get_trainable_parameters()]

traj_text2 = traj_early.to_text()
state2 = tkw2.compress(traj_early)
comp_text2 = TEMPLATE.render(state2)

lp = tkw2.get_log_probs(traj_text2, comp_text2)
(-lp.mean()).backward()
opt.step()

after = [p.data for p in tkw2.get_trainable_parameters()]
changed = sum(1 for b, a in zip(before, after) if not torch.equal(b, a))
print(f"{changed}/{len(before)} parameters changed after one optimizer step. ✓")

In [ ]:
# ── Checkpoint roundtrip ────────────────────────────────────────────────────────
with tempfile.TemporaryDirectory() as tmp:
    # Set a known value, save, reload into a fresh instance, verify
    tkw.get_trainable_parameters()[0].data.fill_(0.42)
    tkw.save_checkpoint(tmp)

    fresh = TrainableKeywordSummaryCompressor(device="cpu")
    fresh.load_checkpoint(tmp)

    loaded_val = fresh.get_trainable_parameters()[0].data.mean().item()
    assert abs(loaded_val - 0.42) < 1e-5, f"Loaded value {loaded_val} != 0.42"
    print(f"Checkpoint roundtrip OK — loaded value ≈ {loaded_val:.4f} ✓")

## Section 5 — Run the automated test suite

The cells below run `pytest` directly from the notebook.
This is faster than switching to a terminal and gives you a clickable test report.

**Files covered:**
- `tests/test_compressor/test_comprehensive.py` — the new comprehensive suite (this notebook)
- `tests/test_compressor/test_template.py` — template render/parse/validate
- `tests/test_compressor/test_dummy_compressor.py` — DummyCompressor unit tests

Slow tests (TransformerCompressor with HuggingFace download) are excluded by default.

In [ ]:
# ── Run the full compressor test suite (fast tests only) ───────────────────────
# -v  : verbose output (test name + pass/fail)
# -x  : stop on first failure (remove to see all failures)
# -m "not slow" : skip TransformerCompressor tests that download HuggingFace weights
import subprocess

result = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        str(REPO_ROOT / "tests" / "test_compressor"),
        "-v",
        "-m", "not slow",
        "--tb=short",   # concise traceback on failure
        "--no-header",
    ],
    capture_output=True,
    text=True,
    cwd=str(REPO_ROOT),
)

print(result.stdout)
if result.returncode != 0:
    print("\n--- STDERR ---")
    print(result.stderr)

In [ ]:
# ── Run ONLY the comprehensive test file ───────────────────────────────────────
# Useful when iterating on a specific compressor implementation.
result2 = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        str(REPO_ROOT / "tests" / "test_compressor" / "test_comprehensive.py"),
        "-v",
        "-m", "not slow",
        "--tb=long",    # full traceback for easier debugging
    ],
    capture_output=True,
    text=True,
    cwd=str(REPO_ROOT),
)

print(result2.stdout)
if result2.returncode != 0:
    print(result2.stderr)

In [ ]:
# ── Run slow tests (TransformerCompressor — downloads flan-t5-small ~300 MB) ───
# Only run this if you have a stable internet connection and disk space.
# Comment out if you're offline or on a metered connection.

RUN_SLOW = False  # ← set to True to enable

if RUN_SLOW:
    result3 = subprocess.run(
        [
            sys.executable, "-m", "pytest",
            str(REPO_ROOT / "tests" / "test_compressor" / "test_comprehensive.py"),
            "-v",
            "-m", "slow",
            "--tb=short",
        ],
        capture_output=True,
        text=True,
        cwd=str(REPO_ROOT),
    )
    print(result3.stdout)
else:
    print("Slow tests skipped (RUN_SLOW=False).")

## Section 6 — Manual context inspection

Run any registered compressor against the four pre-made dev contexts and inspect the
compressed output by eye. This is the most direct way to judge whether a compressor is
extracting the right information from the trajectory.

**What to look for:**

| Context | What a good compressor should do |
|---|---|
| `01_early_episode.json` (2 steps) | Identify hotel search results as the key discovery; flag booking as the next open question |
| `02_mid_episode.json` (6 steps) | Distinguish hotel booking (committed) from restaurant list (exploratory); note the corrected failed call |
| `03_late_episode.json` (8 steps) | Accurate committed-cost tally; itinerary sketch with all booked items; identify day-4 evening as the only open question |
| `04_stress_context.json` (12 steps) | Suppress duplicate hotel search; correct the budget miscalculation; flag duplicate booking attempt as error, NOT decision |

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Set COMPRESSOR_UNDER_TEST to any CompressorBase instance you want to inspect.
# Swap this out to compare your new compressor against the existing baselines.

COMPRESSOR_UNDER_TEST = kw_compressor          # non-trainable baseline
# COMPRESSOR_UNDER_TEST = tkw                  # trainable keyword variant
# COMPRESSOR_UNDER_TEST = IdentityCompressor() # full-trajectory passthrough
# COMPRESSOR_UNDER_TEST = DummyCompressor()    # tiny seq2seq

# Which contexts to inspect (adjust as needed)
CONTEXT_FILES = [
    "01_early_episode.json",
    "02_mid_episode.json",
    "03_late_episode.json",
    "04_stress_context.json",
]

print(f"Compressor: {type(COMPRESSOR_UNDER_TEST).__name__}")
print(f"Metadata  : {COMPRESSOR_UNDER_TEST.get_metadata()}")

In [ ]:
# ── Run all contexts ───────────────────────────────────────────────────────────
# For each context: print the challenge, run the compressor, show the output.
# Assess by eye: did the compressor capture the most important information?

results = {}  # context_file → (state, metadata)

for ctx_file in CONTEXT_FILES:
    traj, meta = load_dev_context(ctx_file)
    print(f"\n{'#'*70}")
    print(f"# Context: {ctx_file}")
    print(f"# {meta['description']}")
    print(f"# Challenge: {meta['compression_challenge'][:120]}...")
    print(f"# Steps: {traj.total_steps}")

    state = COMPRESSOR_UNDER_TEST.compress(traj)
    results[ctx_file] = (state, meta)

    show_compression_result(state, type(COMPRESSOR_UNDER_TEST).__name__)

In [ ]:
# ── Token count comparison table ───────────────────────────────────────────────
# Shows how compressed output size scales with trajectory length.
# A good compressor keeps token_count roughly flat as step count grows.
# The IdentityCompressor will show linear growth — that's expected and bad.

print(f"\n{'Context':<35} {'Steps':>6} {'Tokens':>8} {'Tokens/Step':>12}")
print("-" * 65)
for ctx_file, (state, _) in results.items():
    traj, _ = load_dev_context(ctx_file)
    steps = traj.total_steps
    tokens = state.token_count
    ratio = f"{tokens / max(steps, 1):.1f}" if steps > 0 else "N/A"
    print(f"{ctx_file:<35} {steps:>6} {tokens:>8} {ratio:>12}")

In [ ]:
# ── Side-by-side comparison: your compressor vs IdentityCompressor ─────────────
# Run both on the stress context and print just the DECISIONS_MADE and OPEN_QUESTIONS
# sections for easy comparison.

STRESS_FILE = "04_stress_context.json"
traj_stress, meta_stress = load_dev_context(STRESS_FILE)

identity = IdentityCompressor()
state_identity = identity.compress(traj_stress)
state_custom = COMPRESSOR_UNDER_TEST.compress(traj_stress)

print(f"\nStress context: {meta_stress['description']}")
print(f"\nChallenge: {meta_stress['compression_challenge']}")
print()

for label, state in [("IdentityCompressor", state_identity), (type(COMPRESSOR_UNDER_TEST).__name__, state_custom)]:
    print(f"\n--- {label} ---")
    print(f"  token_count     : {state.token_count}")
    print(f"  decisions_made  : {state.decisions_made}")
    print(f"  open_questions  : {state.open_questions}")
    print(f"  key_discoveries : {state.key_discoveries}")

## Section 7 — Key considerations for compressor development

These are the essential things to keep in mind when building a new compressor.
Only the most impactful points are listed here — not an exhaustive list.

---

### 1. Gradient connectivity is non-negotiable

If `get_log_probs()` returns a tensor that is not connected to the autograd graph,
PPO's policy gradient loss will have zero gradient and training will appear to converge
while doing nothing. This is silent and hard to debug.

**Check:** After every architectural change, re-run Section 4's gradient flow cell.
Ensure every parameter in `get_trainable_parameters()` has a non-None `.grad` after
one `loss.backward()` call.

```python
# Minimum gradient-flow smoke test — run this before committing
lp = compressor.get_log_probs(traj_text, comp_text)
(-lp.mean()).backward()
assert all(p.grad is not None for p in compressor.get_trainable_parameters())
```

---

### 2. The template is a hard constraint — do not bypass it

`CompressedStateTemplate.validate()` enforces that all six sections are non-empty.
The agent's `ContextBuilder` calls `render()` before injecting the state into the
LLM prompt; if a section is empty, the LLM sees a blank section which wastes context
and degrades planning quality.

**Check:** Always call `TEMPLATE.validate(state)` at the end of your `compress()` method
during development. Remove it from production code (it adds overhead) but keep it in tests.

---

### 3. Log-prob values must be ≤ 0

Log-probabilities are the log of values in [0, 1], so they must be non-positive.
PPO computes the importance-sampling ratio as `exp(new_log_prob - old_log_prob)`;
positive log-probs produce ratios > 1 even when the policy has not moved, which
causes incorrect clipping and unstable training.

Common mistakes: using `torch.log_softmax` but forgetting the negative sign, or
returning raw logits instead of log-probabilities.

---

### 4. `CompressorCheckpointError` — not `FileNotFoundError`

The training loop catches `CompressorCheckpointError` specifically to handle resume
logic. If your `load_checkpoint()` raises a generic exception, the trainer will crash
instead of gracefully handling a missing checkpoint.

```python
# Correct pattern
from optimized_llm_planning_memory.core.exceptions import CompressorCheckpointError

def load_checkpoint(self, path: str) -> None:
    p = Path(path)
    if not p.exists():
        raise CompressorCheckpointError(f"Checkpoint not found at {p}")
    try:
        ckpt = torch.load(p, map_location="cpu")
        # ... restore state ...
    except Exception as exc:
        raise CompressorCheckpointError(f"Failed to load: {exc}") from exc
```

---

### 5. Token count efficiency — inspect with the stress context

The point of the compressor is to use *fewer* tokens than the raw trajectory while
preserving planning-relevant information. Check the token count table in Section 6:

- `IdentityCompressor` (baseline): token count grows linearly with steps.
- A good compressor: token count grows sub-linearly or stays roughly flat.
- A bad compressor: smaller token count but loses constraint or booking information.

The stress context (12 steps) is the hardest case — a compressor that looks good on
the early context may fail badly here due to redundant and contradictory steps.

## Section 8 — Registering your compressor

Once your compressor passes all tests and the manual inspection looks good, integrate it
into the rest of the pipeline:

**Step 1: Add a Hydra config file**

Create `configs/compressor/keyword_summary.yaml`:
```yaml
_target_: optimized_llm_planning_memory.compressor.keyword_summary_compressor.TrainableKeywordSummaryCompressor
device: cpu
```

**Step 2: Export from `compressor/__init__.py`**

```python
from .keyword_summary_compressor import KeywordSummaryCompressor, TrainableKeywordSummaryCompressor
```

**Step 3: Run the full integration test suite**

```bash
pytest tests/ -q -m "not slow and not integration"
```

**Step 4: Move the compressor class to its own file**

The class defined in this notebook is for development only. Before production use,
move it to `src/optimized_llm_planning_memory/compressor/keyword_summary_compressor.py`
and add the module docstring and design rationale comment that the project style requires.

## Section 9 — MCTS-Aware Compression (Developer Toggle)

This section lets you run the same compression pipeline with or without Monte Carlo Tree Search (MCTS).
Set `USE_MCTS = True` in the next cell to activate the full MCTS → distillation path.
All subsequent cells guard with `if not USE_MCTS: raise SystemExit("Skipped.")` so you can
safely run the notebook end-to-end with `USE_MCTS = False` to skip this section entirely.

---

### What changes when `USE_MCTS = True`

| Without MCTS | With MCTS |
|---|---|
| Agent picks one action per step (linear ReAct) | Agent explores `branching_factor` candidate actions per node |
| `LLMCompressor.compress(trajectory)` | `LLMMCTSCompressor.compress_with_tree(tree_repr)` |
| `CompressedState` has `top_candidates=None`, `tradeoffs=None` | Both fields populated from the MCTS tree |
| `ContextBuilder` renders the 6 standard sections | Renders standard sections + `[TOP CANDIDATE PLANS]` + `[TRADEOFFS]` blocks |
| `EpisodeLog.mcts_stats = None` | `EpisodeLog.mcts_stats` contains `MCTSStats` |

**Key classes introduced:**

```
MCTSConfig          — hyperparameters (simulations, depth, branching, UCB1 constant)
NodeEvaluator       — scores a trajectory node: heuristic fast path + optional LLM tiebreak
MCTSController      — orchestrates SELECT→EXPAND→SIMULATE→BACKPROP loop
MCTSTreeRepresentation — immutable output: best_path_trajectory, alternatives, top_candidates, tradeoffs, stats
LLMMCTSCompressor   — MCTSAwareCompressor that distils the tree into a richer CompressedState
```

**Design invariant:** `LLMMCTSCompressor.compress_with_tree()` must always pass `TEMPLATE.validate()` —
all 6 standard sections must be non-empty regardless of search depth.

In [ ]:
# ── 9-1  Toggle + conditional imports ──────────────────────────────────────────
# Set USE_MCTS = True to run the full MCTS → compress_with_tree() pipeline.
# Leave False to skip every subsequent cell in this section cleanly.

USE_MCTS = False  # ← change to True to enable

if USE_MCTS:
    from optimized_llm_planning_memory.mcts import (
        MCTSConfig,
        MCTSController,
        MCTSTreeRepresentation,
        NodeEvaluator,
    )
    from optimized_llm_planning_memory.compressor.llm_mcts_compressor import LLMMCTSCompressor
    from optimized_llm_planning_memory.compressor.mcts_aware import MCTSAwareCompressor
    print("MCTS imports OK.")
    print("  MCTSConfig, MCTSController, NodeEvaluator, LLMMCTSCompressor loaded.")
else:
    print("USE_MCTS=False — Section 9 cells will no-op. Set USE_MCTS=True to proceed.")

In [ ]:
# ── 9-2  MCTSConfig + NodeEvaluator + MCTSController ───────────────────────────
if not USE_MCTS:
    raise SystemExit("Skipped — set USE_MCTS=True in cell 9-1 to run.")

# LLM_MODEL_ID: any litellm-compatible model string.
# Use a cheap/fast model during development; swap in a stronger one for final eval.
LLM_MODEL_ID = "openai/gpt-4o-mini"

# MCTSConfig controls the search budget.
# For interactive development: keep num_simulations small (3-5) to avoid long waits.
# For real evaluation: use the react_mcts.yaml defaults (num_simulations=50).
mcts_config = MCTSConfig(
    num_simulations=4,       # ← small for quick iteration; raise for real evaluation
    max_depth=5,
    branching_factor=2,      # candidate actions per expansion
    exploration_constant=1.414,
    temperature=0.7,
    evaluator_model_id=LLM_MODEL_ID,
    use_cached_evaluations=True,  # skip re-scoring identical trajectory snapshots
)

# NodeEvaluator scores tree nodes during SIMULATE phase.
# Two-tier: heuristic fast path first; LLM tiebreak only if score is ambiguous [0.3, 0.7].
evaluator = NodeEvaluator(model_id=LLM_MODEL_ID, config=mcts_config)

# MCTSController orchestrates the full SELECT→EXPAND→SIMULATE→BACKPROP loop.
# It is stateless between search() calls — a fresh MCTSTree is built each time.
controller = MCTSController(
    evaluator=evaluator,
    llm_model_id=LLM_MODEL_ID,
    config=mcts_config,
)

# LLMMCTSCompressor distils the search tree into a CompressedState.
# compress()           → delegates to LLMCompressor (standard non-MCTS path)
# compress_with_tree() → uses tree + LLM to populate all 6 sections + MCTS fields
mcts_compressor = LLMMCTSCompressor(llm_model_id=LLM_MODEL_ID)

print(f"MCTSConfig          : num_simulations={mcts_config.num_simulations}, "
      f"branching_factor={mcts_config.branching_factor}, max_depth={mcts_config.max_depth}")
print(f"NodeEvaluator       : model={LLM_MODEL_ID}, cached={mcts_config.use_cached_evaluations}")
print(f"MCTSController      : ready")
print(f"LLMMCTSCompressor   : ready (method='llm_mcts')")

In [ ]:
# ── 9-3  Run controller.search() on the late-episode dev context ────────────────
# The late-episode context (8 steps) is the most realistic starting point for MCTS:
# enough trajectory history to score branches meaningfully, but not so long that
# the search budget is trivially dominated by a single branch.
if not USE_MCTS:
    raise SystemExit("Skipped.")

from optimized_llm_planning_memory.core.models import UserRequest

traj_late, meta_late = load_dev_context("03_late_episode.json")

print(f"Context  : {meta_late['description']}")
print(f"Challenge: {meta_late['compression_challenge']}")
print(f"Steps    : {traj_late.total_steps}")
print()

# Build a minimal UserRequest so NodeEvaluator can check constraint coverage.
# In production this comes from the agent; here we build a stub from the metadata.
request = UserRequest(
    request_id=traj_late.request_id,
    raw_text=meta_late["request_summary"],
    constraints=[],   # no parsed constraints in dev context metadata
)

# evaluator.set_request() is called by MCTSController.search() internally,
# but we call it here explicitly so NodeEvaluator is ready for standalone testing.
evaluator.set_request(request)

print("Running MCTS search... (this makes LLM calls — expect a few seconds)")
import time
t0 = time.time()
tree_repr = controller.search(
    trajectory=traj_late,
    compressed_state=None,   # no prior compressed state — first compression
    request=request,
)
elapsed = time.time() - t0

print(f"\nSearch complete in {elapsed:.1f}s")
print(f"  nodes_explored      : {tree_repr.stats.nodes_explored}")
print(f"  max_depth_reached   : {tree_repr.stats.max_depth_reached}")
print(f"  num_simulations     : {tree_repr.stats.num_simulations}")
print(f"  root_value          : {tree_repr.stats.root_value:.3f}")
print(f"  avg_branching_factor: {tree_repr.stats.avg_branching_factor:.2f}")
print(f"  top_candidates      : {len(tree_repr.top_candidates)} items")
print(f"  alternative_paths   : {len(tree_repr.alternative_paths)} paths")

In [ ]:
# ── 9-4  Inspect MCTSTreeRepresentation ────────────────────────────────────────
if not USE_MCTS:
    raise SystemExit("Skipped.")

print("── Best path trajectory ─────────────────────────────────────────────────")
best = tree_repr.best_path_trajectory
print(f"  trajectory_id : {best.trajectory_id}")
print(f"  total_steps   : {best.total_steps}")
print(f"  last action   : {best.steps[-1].action.tool_name if best.steps else '(none)'}")

print("\n── Alternative paths ────────────────────────────────────────────────────")
if tree_repr.alternative_paths:
    for i, alt in enumerate(tree_repr.alternative_paths):
        last_action = alt.steps[-1].action.tool_name if alt.steps else "(none)"
        print(f"  [{i+1}] id={alt.trajectory_id[:8]}... steps={alt.total_steps} last={last_action}")
else:
    print("  (none — search only explored one path at this budget)")

print("\n── Top candidates ───────────────────────────────────────────────────────")
for i, cand in enumerate(tree_repr.top_candidates):
    print(f"  {i+1}. {cand}")

print("\n── Tradeoffs ────────────────────────────────────────────────────────────")
print(tree_repr.tradeoffs if tree_repr.tradeoffs else "(empty — populated by compress_with_tree())")

print("\n── MCTSStats (frozen Pydantic) ───────────────────────────────────────────")
print(tree_repr.stats.model_dump_json(indent=2))

In [ ]:
# ── 9-5  compress_with_tree() forward pass + template validation ────────────────
if not USE_MCTS:
    raise SystemExit("Skipped.")

print("Running compress_with_tree()... (one LLM call)")
t1 = time.time()
mcts_state = mcts_compressor.compress_with_tree(tree_repr, previous_state=None)
elapsed2 = time.time() - t1

print(f"Done in {elapsed2:.1f}s\n")

# ── Structural checks ──────────────────────────────────────────────────────────
assert isinstance(mcts_state, CompressedState), "Expected CompressedState"
assert mcts_state.compression_method == "llm_mcts", (
    f"Expected 'llm_mcts', got '{mcts_state.compression_method}'"
)

# Template validation — all 6 standard sections must be non-empty
TEMPLATE.validate(mcts_state)  # raises CompressedStateValidationError if any section empty
print("TEMPLATE.validate() passed — all 6 standard sections populated.")

# MCTS-specific fields
assert mcts_state.top_candidates is not None, "top_candidates should not be None"
assert len(mcts_state.top_candidates) > 0, "top_candidates should have at least one entry"
assert mcts_state.tradeoffs is not None, "tradeoffs should not be None"
assert len(mcts_state.tradeoffs) > 0, "tradeoffs should be a non-empty string"
print(f"MCTS fields populated: {len(mcts_state.top_candidates)} candidates, "
      f"{len(mcts_state.tradeoffs)} chars of tradeoffs.")

# IDs
assert mcts_state.trajectory_id == tree_repr.best_path_trajectory.trajectory_id
assert mcts_state.state_id  # non-empty UUID
print(f"trajectory_id links to best_path_trajectory: OK")

# Show rendered output
show_compression_result(mcts_state, "LLMMCTSCompressor")

In [ ]:
# ── 9-6  Side-by-side: standard CompressedState vs MCTS CompressedState ─────────
# Compare the keyword-summary compressor (Section 2) against LLMMCTSCompressor
# on the same late-episode context to see what the MCTS fields add.
if not USE_MCTS:
    raise SystemExit("Skipped.")

# Run the standard (non-MCTS) compressor on the same trajectory for comparison
standard_state = kw_compressor.compress(traj_late)

print(f"{'='*70}")
print(f"{'STANDARD (KeywordSummaryCompressor)':^70}")
print(f"{'='*70}")
print(f"  compression_method : {standard_state.compression_method}")
print(f"  top_candidates     : {standard_state.top_candidates!r}  ← always None for non-MCTS")
print(f"  tradeoffs          : {standard_state.tradeoffs!r}  ← always None for non-MCTS")
print(f"  token_count        : {standard_state.token_count}")
print()
print("  decisions_made     :")
for d in standard_state.decisions_made:
    print(f"    • {d}")
print()
print("  open_questions     :")
for q in standard_state.open_questions:
    print(f"    • {q}")

print()
print(f"{'='*70}")
print(f"{'MCTS (LLMMCTSCompressor)':^70}")
print(f"{'='*70}")
print(f"  compression_method : {mcts_state.compression_method}")
print(f"  top_candidates     : {len(mcts_state.top_candidates)} entries")
for i, c in enumerate(mcts_state.top_candidates or []):
    print(f"    {i+1}. {c}")
print(f"  tradeoffs          :")
for line in (mcts_state.tradeoffs or "").split("\n"):
    print(f"    {line}")
print(f"  token_count        : {mcts_state.token_count}")
print()
print("  decisions_made     :")
for d in mcts_state.decisions_made:
    print(f"    • {d}")
print()
print("  open_questions     :")
for q in mcts_state.open_questions:
    print(f"    • {q}")

In [ ]:
# ── 9-7  Token count comparison: raw trajectory vs standard vs MCTS ─────────────
# Shows the size of each representation in approximate token units.
# "Approximate" because token_count is computed by whitespace-splitting, not a
# real tokenizer; it is good enough for relative comparisons.
if not USE_MCTS:
    raise SystemExit("Skipped.")

def _word_count(text: str) -> int:
    return len(text.split())

# Raw trajectory text (what a RAW-mode agent sees in its context window)
raw_text = traj_late.to_text()
raw_tokens = _word_count(raw_text)

# Standard compressed state (rendered template, 6 sections only)
std_rendered = TEMPLATE.render(standard_state)
std_tokens = _word_count(std_rendered)

# MCTS compressed state (rendered template + top_candidates + tradeoffs)
mcts_rendered = TEMPLATE.render(mcts_state)
mcts_extra = ""
if mcts_state.top_candidates:
    mcts_extra += "\n[TOP CANDIDATE PLANS FROM SEARCH]\n"
    mcts_extra += "\n".join(f"  {i+1}. {c}" for i, c in enumerate(mcts_state.top_candidates))
if mcts_state.tradeoffs:
    mcts_extra += "\n\n[TRADEOFFS]\n" + mcts_state.tradeoffs
mcts_full = mcts_rendered + mcts_extra
mcts_tokens = _word_count(mcts_full)

# Compression ratios
std_ratio = std_tokens / raw_tokens if raw_tokens else float("inf")
mcts_ratio = mcts_tokens / raw_tokens if raw_tokens else float("inf")

print(f"\n{'Representation':<35} {'~Tokens':>8} {'vs Raw':>10}")
print("-" * 55)
print(f"{'Raw trajectory':<35} {raw_tokens:>8}  {'1.00x':>10}")
print(f"{'Standard compressed (no MCTS)':<35} {std_tokens:>8}  {std_ratio:.2f}x")
print(f"{'MCTS compressed (template+extra)':<35} {mcts_tokens:>8}  {mcts_ratio:.2f}x")
print()
print("Note: MCTS adds top_candidates + tradeoffs to the context window.")
print(f"  Extra MCTS tokens: {mcts_tokens - std_tokens} "
      f"({(mcts_tokens - std_tokens) / raw_tokens * 100:.1f}% of raw)")
print()
print("A good MCTS compressor trades a small token overhead for richer planning")
print("information — the agent sees not just WHERE it is, but WHAT options remain.")

### Developer checklist for `MCTSAwareCompressor` subclasses

Before committing a new MCTS-aware compressor:

**Core contract**
- [ ] Subclasses `MCTSAwareCompressor` (which extends `CompressorBase`)
- [ ] Implements both `compress(trajectory, previous_state)` **and** `compress_with_tree(tree_repr, previous_state)`
- [ ] `compress()` works correctly without any MCTS input — non-MCTS fallback path must be independent
- [ ] `compress_with_tree()` returns `CompressedState(compression_method="llm_mcts", ...)`
- [ ] `compress_with_tree()` passes `TEMPLATE.validate()` — all 6 sections non-empty

**MCTS-specific fields**
- [ ] `top_candidates` is a non-empty `list[str]` after `compress_with_tree()` — not `None`
- [ ] `tradeoffs` is a non-empty `str` after `compress_with_tree()` — not `None`
- [ ] `state_id` is a valid UUID (non-empty string)
- [ ] `trajectory_id` matches `tree_repr.best_path_trajectory.trajectory_id`

**EpisodeLog integration**
- [ ] When used in MCTS_COMPRESSOR mode, `EpisodeLog.mcts_stats` is populated (not `None`)
- [ ] When used in any other mode, `EpisodeLog.mcts_stats` remains `None`

**ContextBuilder integration**
- [ ] `ContextBuilder._history_mcts_compressor()` renders the `[TOP CANDIDATE PLANS FROM SEARCH]`
      block only when `compressed_state.top_candidates` is non-empty
- [ ] `ContextBuilder._history_mcts_compressor()` renders the `[TRADEOFFS]` block only
      when `compressed_state.tradeoffs` is non-empty

**Run the MCTS test suite**
```bash
pytest tests/test_mcts/ -v --tb=short
```

**Run the full suite to confirm no regressions**
```bash
pytest tests/ -v -m "not slow" --tb=short
```